# Task 2 — Notebook 02: Rotation metrics (processed CDL)

1. Build 10-year CDL sequences for pixels **ever** corn or soy in 2013–2022.  
2. Plot **`n_cornsoy_years`** and apply **rotation-eligible** filter: keep pixels with corn/soy in **≥ N years** (default **5** from `rotation_eligibility.min_cornsoy_years_for_metrics` in YAML).  
3. Save **eligible-only** rows to `data/processed/task2/rotation_metrics.parquet`.  
4. **Markov transition matrix** (corn / soy / other) across eligible pixels → heatmap + CSV.  
5. **Transition volume by origin** (corn vs soy vs “other” as share of all year-to-year edges) — calibrates corn-heavy stacks.  
6. **Asymmetry** — ratio **P(soy→corn) / P(corn→soy)** plus a **four-bar** chart of the main corn/soy transition probabilities.  
7. **Max run length** discrete distribution (bars 1–10; ≥7 in red = monoculture rule zone).

*Rationale:* pixels with only 1–2 corn/soy years have too few transitions for meaningful alternation or edit-distance scores.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.cdl_parquet import (
    load_cdl_spatial_metadata,
    load_cdl_wide_years,
    wide_to_label_stack,
    year_range_inclusive,
)
from src.modeling.rotation_classifier import (
    alternation_score_batch,
    cornsoy_years_count_batch,
    crop_share_batch,
    max_run_length_batch,
    pattern_edit_distance_batch,
    shannon_entropy_batch,
    transition_counts_corn_soy_other,
)

with open(REPO_ROOT / "configs" / "task2_crop_rotation.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

MIN_CS = int(cfg.get("rotation_eligibility", {}).get("min_cornsoy_years_for_metrics", 5))
YEARS = year_range_inclusive(cfg["cdl"]["year_range"])
corn = int(cfg["cdl"]["crop_classes"]["corn"])
soy = int(cfg["cdl"]["crop_classes"]["soybean"])
meta = load_cdl_spatial_metadata(REPO_ROOT)
H, W = int(meta["height"]), int(meta["width"])

cdl_df = load_cdl_wide_years(REPO_ROOT, YEARS)
stack = wide_to_label_stack(cdl_df, YEARS, H, W)
data = stack
ever = np.any(np.isin(data, [corn, soy]), axis=0)
iy, ix = np.where(ever)
seqs = data[:, iy, ix].T.astype(np.int16)
n_ever = seqs.shape[0]
print("Pixels ever corn/soy:", f"{n_ever:,}")

ncs = cornsoy_years_count_batch(seqs)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ncs, bins=range(0, 12), color="gray", edgecolor="k", alpha=0.75)
ax.axvline(MIN_CS, color="crimson", lw=2, label=f"Eligibility cutoff (≥{MIN_CS} corn/soy years)")
ax.set_xlabel("n_cornsoy_years (of 10)")
ax.set_ylabel("Pixel count")
ax.set_title("Corn/soy year count — ever-corn/soy pool")
ax.legend()
fig.tight_layout()
fig_dir = REPO_ROOT / cfg["output"]["figures_dir"]
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "task2__ncornsoy_histogram.png", dpi=200, bbox_inches="tight")
plt.show()

elig = ncs >= MIN_CS
seqs_el = seqs[elig]
iy_el, ix_el = iy[elig], ix[elig]
print("Rotation-eligible pixels (≥ MIN_CS corn/soy years):", f"{seqs_el.shape[0]:,}", f"({100*seqs_el.shape[0]/max(n_ever,1):.1f}% of ever pool)")

alt = alternation_score_batch(seqs_el)
run = max_run_length_batch(seqs_el)
dist = pattern_edit_distance_batch(seqs_el, mask_to_cornsoy=True)
ncs_el = cornsoy_years_count_batch(seqs_el)
share = crop_share_batch(seqs_el)
ent = shannon_entropy_batch(seqs_el)

metrics_df = pd.DataFrame(
    {
        "iy": iy_el.astype(np.int32),
        "ix": ix_el.astype(np.int32),
        "alternation_score": alt,
        "max_run_length": run,
        "pattern_edit_distance": dist,
        "entropy": ent,
        "n_cornsoy_years": ncs_el,
        "crop_share": share,
    }
)

M_count, M_prob = transition_counts_corn_soy_other(seqs_el, corn=corn, soy=soy)
labels = ["corn", "soy", "other"]
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(M_prob, cmap="YlOrRd", vmin=0.0, vmax=1.0)
ax.set_xticks(range(3), labels=labels)
ax.set_yticks(range(3), labels=labels)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{M_prob[i, j]:.2f}", ha="center", va="center", color="black", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xlabel("To (year t+1)")
ax.set_ylabel("From (year t)")
ax.set_title("Markov P(to|from) — rotation-eligible pixels")
fig.tight_layout()
fig.savefig(fig_dir / "task2__markov_corn_soy_other.png", dpi=200, bbox_inches="tight")
plt.show()

out_dir = REPO_ROOT / cfg["output"]["processed_dir"]
out_dir.mkdir(parents=True, exist_ok=True)
metrics_df.to_parquet(out_dir / "rotation_metrics.parquet", index=False)
pd.DataFrame(M_count, index=labels, columns=labels).to_csv(out_dir / "task2__markov_transition_counts.csv")
pd.DataFrame(M_prob, index=labels, columns=labels).to_csv(out_dir / "task2__markov_transition_probs.csv")
print("Wrote rotation_metrics.parquet rows", len(metrics_df))
print("Markov counts sum", float(M_count.sum()))


Pixels ever corn/soy: 530,472


C:\Users\sardo\AppData\Local\Temp\ipykernel_25076\2748468590.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Rotation-eligible pixels (≥ MIN_CS corn/soy years): 301,485 (56.8% of ever pool)


Wrote rotation_metrics.parquet rows 301485
Markov counts sum 2713365.0


C:\Users\sardo\AppData\Local\Temp\ipykernel_25076\2748468590.py:107: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [2]:
row_tot = M_count.sum(axis=1)
grand = float(row_tot.sum())
print("Share of year-to-year transitions by origin state (row sums of Markov counts):")
for i, lab in enumerate(labels):
    print(f"  {lab}: {int(row_tot[i]):,}  ({100.0 * float(row_tot[i]) / grand:.2f}%)")
print(f"Total transitions: {int(grand):,}  (= {len(metrics_df):,} eligible pixels × 9 adjacent pairs)")


Share of year-to-year transitions by origin state (row sums of Markov counts):
  corn: 1,660,603  (61.20%)
  soy: 611,458  (22.54%)
  other: 441,304  (16.26%)
Total transitions: 2,713,365  (= 301,485 eligible pixels × 9 adjacent pairs)


In [3]:
p_cs = float(M_prob[0, 1])
p_sc = float(M_prob[1, 0])
asym = p_sc / max(p_cs, 1e-12)
print(f"P(corn→soy)={p_cs:.3f}  P(soy→corn)={p_sc:.3f}  asymmetry ratio P(S→C)/P(C→S) = {asym:.2f}×")

fig, ax = plt.subplots(figsize=(6.5, 4))
bar_names = ["P(C→C)", "P(C→S)", "P(S→C)", "P(S→S)"]
bar_vals = [float(M_prob[0, 0]), p_cs, p_sc, float(M_prob[1, 1])]
bar_cols = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12"]
ax.bar(bar_names, bar_vals, color=bar_cols, edgecolor="k", alpha=0.9)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Probability")
ax.set_title("Key corn/soy Markov transitions (rotation-eligible pixels)")
fig.tight_layout()
fig.savefig(fig_dir / "task2__transition_asymmetry.png", dpi=200, bbox_inches="tight")
plt.show()


P(corn→soy)=0.314  P(soy→corn)=0.764  asymmetry ratio P(S→C)/P(C→S) = 2.44×


C:\Users\sardo\AppData\Local\Temp\ipykernel_25076\255522612.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
runs = metrics_df["max_run_length"].to_numpy(dtype=np.int32)
fig, ax = plt.subplots(figsize=(8, 4))
for r in range(1, 11):
    cnt = int(np.sum(runs == r))
    col = "#c0392b" if r >= 7 else "steelblue"
    ax.bar(r, cnt, width=0.85, color=col, edgecolor="k", alpha=0.88)
ax.set_xticks(range(1, 11))
ax.set_xlabel("Max run length (years)")
ax.set_ylabel("Pixel count")
ax.axvline(6.5, color="gray", linestyle="--", linewidth=1, alpha=0.85)
ax.set_title("Max run length (eligible pixels) — red: run ≥ 7 (monoculture rule)")
fig.tight_layout()
fig.savefig(fig_dir / "task2__runlength_distribution.png", dpi=200, bbox_inches="tight")
plt.show()


C:\Users\sardo\AppData\Local\Temp\ipykernel_25076\831581311.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].hist(metrics_df["alternation_score"], bins=40, color="steelblue", edgecolor="k", alpha=0.7)
axes[0, 0].set_title("Alternation score (eligible)")
axes[0, 1].hist(metrics_df["pattern_edit_distance"], bins=range(0, 12), color="coral", edgecolor="k", alpha=0.7)
axes[0, 1].set_title("Pattern edit distance")
axes[1, 0].hist(metrics_df["max_run_length"], bins=20, color="seagreen", edgecolor="k", alpha=0.7)
axes[1, 0].set_title("Max run length")
axes[1, 1].hist(metrics_df["entropy"], bins=30, color="purple", edgecolor="k", alpha=0.7)
axes[1, 1].set_title("Shannon entropy (bits)")
fig.suptitle("Rotation metrics — rotation-eligible pixels only")
fig.tight_layout()
fig_dir = REPO_ROOT / cfg["output"]["figures_dir"]
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "task2__metric_histograms.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    metrics_df["alternation_score"],
    metrics_df["pattern_edit_distance"],
    c=metrics_df["entropy"],
    s=2,
    alpha=0.25,
    cmap="viridis",
)
plt.colorbar(sc, ax=ax, label="entropy")
ax.set_xlabel("Alternation score")
ax.set_ylabel("Pattern edit distance")
ax.set_title("Metric cross-plot (eligible pixels)")
fig.tight_layout()
fig.savefig(fig_dir / "task2__alt_vs_distance.png", dpi=200, bbox_inches="tight")
plt.show()


C:\Users\sardo\AppData\Local\Temp\ipykernel_25076\2244895883.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\sardo\AppData\Local\Temp\ipykernel_25076\2244895883.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
